# Lab: Flagging Applications That Need Rework

## The problem
Every week the **Government Affairs (GA) office** submits hundreds of applications to Saudi government portals — visa and iqama renewals, family permits, exit/re-entry requests, dependant registrations — on behalf of KAUST faculty, employees and students.

Some applications come back **rejected or returned for correction**. Each rework cycle costs days and a frustrated applicant. If we could **predict in advance which applications are likely to need rework**, an officer could double-check those *before* submission and fix the problem on the spot.

In this lab you'll **train a machine-learning model to flag risky applications** so the team can focus their pre-submission review where it matters most.

## What you'll do
1. Look at a (synthetic) dataset of 2,000 GA applications.
2. Train a model to predict whether each application will need rework.
3. Switch between two different models and compare their accuracy.
4. Move a slider to tune how cautious the system is, and watch the trade-off live.

> **You don't need to write any code.** Just press the ▶ Run button on each cell from top to bottom and read the output.

---


## Step 1 — Load the tools

These are standard data-science libraries (already pre-installed in Google Colab). Just run this cell.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score

from ipywidgets import interact, FloatSlider

np.random.seed(42)
print("Tools loaded.")


## Step 2 — Generate a synthetic GA application dataset

For this lab we create a realistic-looking dataset of **2,000 applications** that the GA office submitted to Saudi ministries. Roughly **15%** of them needed rework (rejected or returned for correction).

| Column | Meaning |
|---|---|
| `applicant_type` | faculty, employee, student, or dependant |
| `application_type` | iqama_renewal, visa, family_permit, exit_reentry |
| `documents_attached` | Number of supporting documents uploaded |
| `days_until_deadline` | How close to the deadline the application was submitted |
| `applicant_tenure_days` | How long the applicant has been at KAUST |
| `prior_applications` | How many past applications this applicant has submitted |
| `weekend_submission` | 1 if submitted on Friday or Saturday, else 0 |
| `complexity_score` | Internal score from 0 (simple) to 5 (complex case) |
| `needs_rework` | **1 = was returned/rejected, 0 = approved on first submission** — what we want to predict |


In [ ]:
def generate_applications(n=2000, rework_rate=0.15, seed=42):
    """Generate a synthetic GA-application dataset.

    The clean and rework groups *overlap* on every numeric feature: rework cases
    only have a *higher chance* of e.g. fewer documents or being last-minute.
    This keeps the problem hard enough that precision/recall < 100% and the
    threshold slider matters.
    """
    rng = np.random.default_rng(seed)
    n_rework = int(n * rework_rate)
    n_clean  = n - n_rework

    # ---- Approved on first submission (the common case) ----
    clean = pd.DataFrame({
        "applicant_type": rng.choice(
            ["faculty", "employee", "student", "dependant"],
            n_clean, p=[0.20, 0.30, 0.35, 0.15]),
        "application_type": rng.choice(
            ["iqama_renewal", "visa", "family_permit", "exit_reentry"],
            n_clean, p=[0.40, 0.20, 0.20, 0.20]),
        "documents_attached": rng.integers(2, 10, n_clean),                    # 2..9
        "days_until_deadline": rng.integers(1, 60, n_clean),                   # 1..59
        "applicant_tenure_days": rng.integers(60, 4000, n_clean),
        "prior_applications": rng.poisson(6, n_clean),
        "weekend_submission": rng.choice([0, 1], n_clean, p=[0.90, 0.10]),
        "complexity_score": rng.integers(0, 6, n_clean),                       # 0..5
        "needs_rework": 0,
    })

    # ---- Required rework: same ranges, but skewed toward "risky" patterns ----
    rework = pd.DataFrame({
        "applicant_type": rng.choice(
            ["faculty", "employee", "student", "dependant"],
            n_rework, p=[0.15, 0.20, 0.30, 0.35]),
        "application_type": rng.choice(
            ["iqama_renewal", "visa", "family_permit", "exit_reentry"],
            n_rework, p=[0.25, 0.30, 0.30, 0.15]),
        "documents_attached": rng.integers(1, 8, n_rework),                    # 1..7 (overlaps)
        "days_until_deadline": rng.integers(0, 30, n_rework),                  # 0..29 (overlaps)
        "applicant_tenure_days": rng.integers(30, 3000, n_rework),
        "prior_applications": rng.poisson(3, n_rework),
        "weekend_submission": rng.choice([0, 1], n_rework, p=[0.70, 0.30]),
        "complexity_score": rng.integers(1, 6, n_rework),                      # 1..5 (overlaps)
        "needs_rework": 1,
    })

    df = pd.concat([clean, rework], ignore_index=True)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    return df

df = generate_applications()
print(f"Total applications: {len(df):,}")
print(f"Needed rework:      {df['needs_rework'].sum():,}  "
      f"({df['needs_rework'].mean()*100:.1f}%)")
df.head()


## Step 3 — A first look at the data

How do applications break down by type, and how many of each needed rework?

In [ ]:
#@title Plot: applications by type — first time vs needed rework
counts = df.groupby(["application_type", "needs_rework"]).size().unstack(fill_value=0)
counts.columns = ["Approved first time", "Needed rework"]

ax = counts.plot(kind="bar", figsize=(8, 4), color=["#4c8bf5", "#e63946"])
ax.set_title("Applications by type — approved first time vs needed rework")
ax.set_ylabel("Number of applications")
ax.set_xlabel("Application type")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## Step 4 — Prepare the data for the model

Models need numbers, not text. We convert the `applicant_type` and `application_type` columns into 0/1 columns, one per category — this is called *one-hot encoding*.

Then we split the data: **80% to train the model**, **20% kept aside as a test set the model has never seen**, so we can measure its real accuracy fairly.

In [ ]:
X = pd.get_dummies(
    df.drop(columns=["needs_rework"]),
    columns=["applicant_type", "application_type"],
    dtype=int,
)
y = df["needs_rework"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Training applications: {len(X_train):,}")
print(f"Test applications:     {len(X_test):,}")


## Step 5 — Train a model and evaluate it

Pick a model from the dropdown on the right of the cell below, then run it.

- **Logistic Regression** — a simple, fast model; easy to explain.
- **Random Forest** — an ensemble of decision trees; usually more accurate, harder to interpret.

The cell will train the model on 1,600 historical applications, then test it on 400 unseen applications and show how well it spots the ones that needed rework.

In [ ]:
#@title Pick a model and run this cell { run: "auto" }
model_choice = "Logistic Regression" #@param ["Logistic Regression", "Random Forest"]

if model_choice == "Logistic Regression":
    model = LogisticRegression(class_weight="balanced", max_iter=1000)
    model.fit(X_train_s, y_train)
    proba_test = model.predict_proba(X_test_s)[:, 1]
else:
    model = RandomForestClassifier(
        n_estimators=200, class_weight="balanced", random_state=42
    )
    model.fit(X_train, y_train)
    proba_test = model.predict_proba(X_test)[:, 1]

preds = (proba_test >= 0.5).astype(int)
cm = confusion_matrix(y_test, preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
prec = precision_score(y_test, preds, zero_division=0)
rec  = recall_score(y_test, preds, zero_division=0)

print(f"Model: {model_choice}")
print(f"Decision threshold: 0.50")
print()
print("On 400 unseen test applications:")
print(f"  Clean apps correctly cleared:        {tn}")
print(f"  Clean apps falsely flagged:          {fp}")
print(f"  Risky apps missed:                   {fn}")
print(f"  Risky apps correctly flagged:        {tp}")
print()
print(f"Precision: {prec*100:5.1f}%   ->  of every 100 apps we flag, ~{prec*100:.0f} actually needed rework")
print(f"Recall:    {rec*100:5.1f}%   ->  we catch {rec*100:.0f}% of the risky apps in the test set")


## Step 6 — The reviewer's dial: precision vs recall

Behind the scenes, the model gives every application a **risk score between 0 and 1**. We choose a *threshold* above which an application is flagged for an extra pre-submission review.

Move the slider below to change that threshold:

- **Low threshold (e.g. 0.20)** → flag many applications → catch more rework cases, but reviewers are flooded with false alarms.
- **High threshold (e.g. 0.80)** → flag few applications → reviewers only see the riskiest ones, but more rework cases slip through.

This is the core **policy choice** when deploying a tool like this: how much pre-submission review capacity do we have, and how cautious do we want to be?

In [ ]:
#@title Interactive: move the slider to change the decision threshold
def show_threshold(threshold=0.5):
    preds = (proba_test >= threshold).astype(int)
    flagged = int(preds.sum())
    cm = confusion_matrix(y_test, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    prec = precision_score(y_test, preds, zero_division=0)
    rec  = recall_score(y_test, preds, zero_division=0)

    print(f"Threshold: {threshold:.2f}")
    print(f"Applications flagged for review: {flagged} of {len(proba_test)} "
          f"({flagged/len(proba_test)*100:.1f}%)")
    print(f"  -> {tp} risky apps caught,  {fp} false alarms,  {fn} risky apps missed")
    print(f"Precision: {prec*100:5.1f}%     Recall: {rec*100:5.1f}%")

interact(show_threshold,
         threshold=FloatSlider(min=0.10, max=0.90, step=0.05, value=0.50,
                               description="Threshold"));


## Step 7 — Putting the results together

The slider above let you explore one threshold at a time. The two charts below summarise the **whole picture** of what the model is doing on the test set:

- **Left:** how precision, recall and the fraction of applications flagged change as we slide the threshold from very lenient to very strict.
- **Right:** the actual *risk scores* the model assigned to every application, coloured by whether the application really needed rework. A model that separates the two colours well is a useful model.


In [ ]:
#@title Plot: model behaviour across thresholds + risk-score distribution
ts = np.linspace(0.02, 0.98, 49)
precs = [precision_score(y_test, (proba_test >= t).astype(int), zero_division=0) * 100 for t in ts]
recs  = [recall_score   (y_test, (proba_test >= t).astype(int), zero_division=0) * 100 for t in ts]
flags = [(proba_test >= t).mean() * 100 for t in ts]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.plot(ts, precs, label="Precision (% of flagged that were really risky)", color="#1f4e79", linewidth=2)
ax.plot(ts, recs,  label="Recall (% of risky apps we caught)",              color="#2a9d8f", linewidth=2)
ax.plot(ts, flags, label="Fraction of all apps flagged",                    color="#e63946", linewidth=2, linestyle="--")
ax.axvline(0.5, color="gray", linestyle=":", alpha=0.7)
ax.text(0.51, 4, "default\n0.50", fontsize=9, color="gray")
ax.set_title("How the model behaves at every threshold")
ax.set_xlabel("Threshold"); ax.set_ylabel("Percentage")
ax.set_ylim(0, 105); ax.set_xlim(0, 1)
ax.legend(loc="lower left", fontsize=8.5)
ax.grid(alpha=0.2)

ax = axes[1]
ax.hist(proba_test[y_test == 0], bins=20, alpha=0.75,
        label="Approved first time", color="#4c8bf5", edgecolor="white")
ax.hist(proba_test[y_test == 1], bins=20, alpha=0.75,
        label="Needed rework",       color="#e63946", edgecolor="white")
ax.axvline(0.5, color="gray", linestyle=":", alpha=0.7)
ax.set_title("Risk score distribution on the 400 test applications")
ax.set_xlabel("Risk score the model assigned"); ax.set_ylabel("Number of applications")
ax.legend(); ax.grid(alpha=0.2)

plt.tight_layout(); plt.show()


**How to read these charts**

- *Left:* the lines cross around 0.5–0.6 — that's the threshold zone where precision and recall are balanced. Move left and you flag more apps (red dashed line goes up) but precision drops. Move right and precision improves but recall (the green line) collapses.
- *Right:* a *useful* model produces a clearly bimodal distribution — most legitimate apps clustered near 0, most risky ones near 1, and only a small overlapping middle. The middle is where the threshold choice actually matters.

If the right-hand histogram had been a single overlapping blob in the middle, the model wouldn't be ready for production no matter what threshold we picked.


## What this means for the GA office

- A model like this **does not replace reviewers** — it directs their attention to the riskiest applications.
- The threshold is a **policy choice**: it depends on review capacity and how much risk of rework the office is willing to absorb.
- A real deployment would need:
  - **Real historical labels** of which past applications needed rework (often the hardest part to assemble).
  - **Regular re-training** as ministry rules and portals change.
  - **Fairness checks** — the model must not unfairly flag specific applicant groups.
  - **A clear human-in-the-loop process**: the model *flags*, an officer *decides*.

You've now trained, compared and tuned a real-style ML model end to end — in under 30 minutes, without writing any code.